# Figures for the SI
## Do not change any of the base code in here, consult with F.H. Garcia before making any changes

In [ ]:
%matplotlib inline
import numpy as np
import scipy
import statistics
import matplotlib as mpl
from matplotlib import gridspec
import matplotlib.ticker as ticker
from scipy.optimize import curve_fit
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator, FormatStrFormatter)
from scipy import interpolate
import matplotlib.patches as mpatches
import pandas as pd 
from numpy import *
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import find_peaks
import matplotlib.ticker as plticker
# import probfit
from scipy import special
import time
import datetime

mpl.rc('font', family='Arial')

In [ ]:
def readPSDData(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=1)
    tmp.columns = ['Energy', 'PSD']
    return tmp

def readFOM(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=1)
    tmp.columns = ['i', 'mu1', 'sigma1', 'a1', 'mu2', 'sigma2', 'a2', 'fom', 'e_lower', 'e_higher']
    return tmp

def readNSpectrum(filename):
    tmp = pd.read_csv(filename, sep = ',', header = None, skiprows=2)
    tmp.columns = ['lightOutput(keVee)-sim', 'Sim-NormCounts', 'lightOutput(keVee)-exp', 'Exp-NormCounts']
    return tmp


In [ ]:
def readNeutronData(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=1)
    tmp.columns = ['binMid', 'binTime(s)', 'Neutrons(cps)','delta-neutrons', 'BackgroundGamma(cps)', 'delta-gamma']
    tmp['binTime(m)'] = tmp['binTime(s)']/60
    return tmp

def readVapor(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=0)
    return tmp

In [ ]:
id419_5sigma = readNeutronData('../DataToPlot/ID419/ID-419_data_300s_bin.csv')
id423_5sigma = readNeutronData('../DataToPlot/ID423/ID-423_data_300s_bin.csv')

id419_4sigma = readNeutronData('../DataToPlot/ID419/ID-419_data_60s_bin-4sigma.csv')
id423_4sigma = readNeutronData('../DataToPlot/ID423/ID-423_data_60s_bin-4sigma.csv')

id419_3sigma = readNeutronData('../DataToPlot/ID419/ID-419_data_300s_bin-3sigma.csv')
id423_3sigma = readNeutronData('../DataToPlot/ID423/ID-423_data_300s_bin-3sigma.csv')

In [ ]:
psdData = readPSDData('Archive/psd_data.csv')
FOM = readFOM('Archive/fom_analysis.csv')
data = readVapor('../DataToPlot/VaporWaveIsland.csv')

In [ ]:
lowBound = FOM['mu1']+5*FOM['sigma1']
FOM['lowBound'] = lowBound
highBound = FOM['lowBound'] + 0.2
FOM['highBound'] = highBound

In [ ]:
newFOM = FOM[1:]

In [ ]:
neutron_lb = savgol_filter(FOM["mu1"] + 5  * FOM["sigma1"], window_length=21, polyorder=3)  # reduce noise
neutron_lb_fit = interpolate.interp1d(FOM['e_lower'], neutron_lb, fill_value=(neutron_lb[0], neutron_lb[-1]), bounds_error=False)  # now it's a function!

In [ ]:
xaxis = np.linspace(0,1663,1000)
yaxis = neutron_lb_fit(xaxis)

# Neutron energy spectrum

In [ ]:
neutronEData = readNSpectrum('ExpVsSimData-Nenergy.csv')
neutronEData.head()

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (8,5))
fig.tight_layout()

axs.plot(neutronEData['lightOutput(keVee)-exp'], neutronEData['Exp-NormCounts'], ls ='', marker = 'o', ms = 5, color ='#424242FF' )
axs.step(neutronEData['lightOutput(keVee)-sim'], neutronEData['Sim-NormCounts'], ls ='-')

# axs.set_title('Beam Loading', fontsize = 16)
axs.set_xlim(-20, 1000)
# axs.set_ylim(-5,180)
axs.set_ylabel('Normalized counts', fontsize=14)
# axs.set_yscale("log")
axs.set_xlabel('Light output (keVee)', fontsize=14)
# axs.axhline(64, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(45, color = 'black', ls = "--", alpha = 0.7)
axs.tick_params(axis="x", labelsize = 12)
axs.annotate('Experiment',(700,0.15), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
axs.annotate('Simulation (with resolution)',(300,0.0), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, color = 'C0' )

plt.savefig("SI-Fig6-NeutronEnergySpectrum.pdf", format="pdf", bbox_inches="tight")
# plt.savefig("SI-Fig6-NeutronEnergySpectrum.png", format="png", bbox_inches="tight")

plt.show()


# Beamloading experiments

In [ ]:
id444 = readNeutronData('Fig3a/ID-419_data_300s_bin.csv')

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (8,5))
fig.tight_layout()

axs.plot(neutronEData['lightOutput'], neutronEData['Normalized'], ls ='', marker = 'o', ms = 5)
axs.set_title('Beam Loading', fontsize = 16)
axs.set_xlim(-0.03, 1)
# axs.set_ylim(-5,180)
axs.set_ylabel('Counts (norm.)', fontsize=14)
axs.set_xlabel('Light output (MeVee)', fontsize=14)
# axs.axhline(64, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(45, color = 'black', ls = "--", alpha = 0.7)
axs.tick_params(axis="x", labelsize = 12)

plt.show()


# SI PSD plot results

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (9,7))
fig.tight_layout(pad=2)
threshold = 50
otherThreshold = 640
axs.plot(xaxis[16:], yaxis[16:],  ls ='-', color = 'red', lw = 1)
axs.plot(xaxis[16:], yaxis[16:]+0.2,  ls ='-', color = 'red', lw = 1)
axs.fill_between(xaxis, yaxis,yaxis+0.2, where=xaxis>threshold, color = 'red', alpha = 0.2)
# axs.plot(psdData['Energy'], psdData['PSD'], ls ='', marker = 'o', ms = 2, alpha = 0.1, color = 'black')
histresult = axs.hist2d(psdData['Energy'], psdData['PSD'], bins = 300, cmin=2, cmap = 'viridis')
img = histresult[3]

axs.set_xlim(0,1650)
axs.set_ylim(0,0.5)
axs.set_ylabel('Pulse shape discrimination (arb. unit)', fontsize=16)

axs.annotate('Gamma-ray channel',(150,0.15), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 16, fontname = 'Arial',color = 'white' )
axs.set_xlabel('Light output (keVee)', fontsize=16)
# axs[0].axhline(0.35, color = 'black', ls = "--", alpha = 0.7)
# axs.axhline(0.16, color = 'black', ls = "--", alpha = 0.7)
# axs.axvline(195,color = 'black', ls ='--')
# axs.axvline(210,color = 'black', ls ='--')
# axs[0].axvline(100, color = 'black')


axs.tick_params(axis="x", labelsize = 14)
axs.tick_params(axis="y", labelsize = 14)
cbar = fig.colorbar(img, ax = axs)
cbar.set_label('Counts', fontsize = 16, rotation = 270, labelpad = 15)
cbar.ax.tick_params(labelsize = 14)

# axs.axvline(195, color = 'red')
# axs.axvline(205, color = 'red')

axs.annotate('Neutron channel',(150,0.325), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 16, fontname = 'Arial', color = 'white' )
plt.savefig("SigmaComp.pdf", format="pdf", bbox_inches="tight")

plt.show()


# Neutron window plot

In [ ]:
fig, ax1 = plt.subplots(figsize = (9,9))

# These are in unitless percentages of the figure size. (0,0 is bottom left)
left, bottom, width, height = [0.58,0.56, 0.3, 0.3]
# ax2 = fig.add_axes([left,bottom, width, height])

ax1.plot(psdData['Energy'], psdData['PSD'], ls ='', marker = 'o', ms = 2, alpha = 0.1, color = 'black')
ax1.set_ylabel('PSD (a. u.)', fontsize=20)
ax1.annotate('Neutron channel',(80,0.325), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 20, fontname = 'Arial', color = 'white' )
ax1.annotate('Gamma-ray channel',(80,0.150), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 20, fontname = 'Arial',color = 'white' )
ax1.set_xlabel('Energy (keVee)', fontsize=20)
ax1.tick_params(axis="x", labelsize = 18)
ax1.tick_params(axis="y", labelsize = 18)

# ax1.plot(xaxis, yaxis,  ls ='--', color = 'C1')
# ax1.plot(FOM['e_lower'], FOM['lowBound'], marker ='o', ls ='')
# ax1.plot(xaxis, yaxis+0.2,  ls ='--', color = 'C1')

# ax1.axvline(195, ls = '--', color = 'C0')
# ax1.axvline(640, ls = '--', color = 'C0')
threshold = 195
otherThreshold = 640
ax1.set_ylim(-0.015,0.5)
ax1.set_xlim(-10,1600)
# ax1.axhline(0.155, color = 'white')
# ax1.fill_between(xaxis, yaxis,yaxis+0.2, where=xaxis>threshold, color = 'C1', alpha = 0.3)
# ax1.fill_between(xaxis, yaxis,yaxis+0.2, where=xaxis>otherThreshold, color = 'white')


# ax2.plot(newFOM['i']*15, newFOM['fom'], ls ='-', marker = 's', ms = 0, color = 'black')
# ax2.set_xlabel('Energy (keVee)', fontsize = 16)
# ax2.set_ylabel('FOM', fontsize = 16)
# ax2.axhline(1.27, ls = '--', color = 'C0')
# ax2.annotate('FOM = 1.27',(50,1.29), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, fontname = 'Arial',color = 'black' )
# ax2.set_xlim(-10, 600)
# ax2.set_ylim(0.4, 2)
# ax2.tick_params(axis="x", labelsize = 14)
# ax2.tick_params(axis="y", labelsize = 14)

plt.show()

# OLD

-------
# Comparing the 5sigma and 4sigma neutron rates

In [ ]:
fig, axs = plt.subplots(1, 3, figsize = (20,7))
fig.tight_layout()

axs[0].errorbar(id419_5sigma['binTime(s)'], id419_5sigma['Neutrons(cps)'], id419_5sigma['delta-neutrons'], ls ='', marker = 'o', ms = 4, capsize = 4, label = 'Beam loaded - 5$\sigma$', color ='#424242FF' )
axs[0].errorbar(id423_5sigma['binTime(s)'], id423_5sigma['Neutrons(cps)'], id423_5sigma['delta-neutrons'], ls ='', marker = 's', ms = 4, capsize = 4, label = 'Electrochemically loaded - 5$\sigma$', color ='#941100FF' )
# axs[0].legend(fontsize = 14, loc = 'lower right')
axs[0].set_title('Target = A5- 5$\sigma$', fontsize = 16)
axs[0].tick_params(axis="y", labelsize = 14)
axs[0].set_xlim(-240,7500)
axs[0].tick_params(axis="x", labelsize = 14)
axs[0].set_ylim(-5,200)
axs[0].xaxis.set_minor_locator(AutoMinorLocator())
axs[0].yaxis.set_minor_locator(AutoMinorLocator())
axs[0].set_ylabel('Neutron rate (1/s)', fontsize=16)
axs[0].axhline(132, color = 'black', ls = "--", alpha = 0.7)
axs[0].axhline(155, color = 'black', ls = "--", alpha = 0.7)
axs[0].axvline(3900, color = 'black', ls = "-", alpha = 0.7)
# axs[0].annotate('E-cell start \n15% increase',(4000,115), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
axs[0].annotate('15% increase',(3900,158), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
loc = plticker.MultipleLocator(base=2000) # this locator puts ticks at regular intervals
axs[0].xaxis.set_major_locator(loc)
axs[0].axvspan(-250,3900, alpha=0.1, color='#4574A2FF')


axs[1].errorbar(id419_4sigma['binTime(s)'], id419_4sigma['Neutrons(cps)'], id419_4sigma['delta-neutrons'], ls ='', marker = 'o', ms = 4, capsize = 4, label = 'Beam loaded - 4$\sigma$', color ='#424242FF' )
axs[1].errorbar(id423_4sigma['binTime(s)'], id423_4sigma['Neutrons(cps)'], id423_4sigma['delta-neutrons'], ls ='', marker = 's', ms = 4, capsize = 4, label = 'Electrochemically loaded - 4$\sigma$', color ='#941100FF' )
axs[1].axhline(141, color = 'black', ls = "--", alpha = 0.7)
axs[1].axhline(166, color = 'black', ls = "--", alpha = 0.7)
axs[1].set_title('Target = A5 - 4$\sigma$', fontsize = 16)
axs[1].xaxis.set_minor_locator(AutoMinorLocator())
axs[1].yaxis.set_minor_locator(AutoMinorLocator())
# axs[1].tick_params(axis="y")
axs[1].set_ylim(-5,200)
axs[1].set_xlim(-240,7500)
# axs[1].legend(fontsize = 14, loc = 'lower right')
axs[1].set_xlabel('Time (s)', fontsize = 16)
# axs[1].annotate('E-cell start \n11% increase',(4300,120), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
# axs[1].annotate('11% increase',(4210,159), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
axs[1].axvline(3900, color = 'black', ls = "-", alpha = 0.7)
# axs[1] = axs[1].twinx()
axs[1].annotate('14% increase',(3900,170), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
axs[1].tick_params(axis="y")
axs[1].tick_params(axis="x", labelsize = 14)
axs[1].xaxis.set_major_locator(loc)
axs[1].axvspan(-250,3900, alpha=0.1, color='#4574A2FF')


axs[2].errorbar(id419_3sigma['binTime(s)'], id419_3sigma['Neutrons(cps)'], id419_3sigma['delta-neutrons'], ls ='', marker = 's', ms = 4, capsize = 4, label = 'Beam loaded - 3$\sigma$', color = '#424242FF')
axs[2].errorbar(id423_3sigma['binTime(s)'], id423_3sigma['Neutrons(cps)'], id423_3sigma['delta-neutrons'], ls ='', marker = 's', ms = 4, capsize = 4, label = 'Electrochemically loaded - 3$\sigma$', color = '#941100FF')
# axs[2].legend(fontsize = 14, loc = 'lower right')
axs[2].set_title('Target = A5 - 3$\sigma$', fontsize = 16)
axs[2].set_ylim(-5,200)
axs[2].xaxis.set_minor_locator(AutoMinorLocator())
axs[2].yaxis.set_minor_locator(AutoMinorLocator())
axs[2].set_xlim(-240,7500)
axs[2].tick_params(axis="x", labelsize = 14)
axs[2].axhline(144, color = 'black', ls = "--", alpha = 0.7)
axs[2].axhline(168, color = 'black', ls = "--", alpha = 0.7)
axs[2].axvline(3960, color = 'black', ls = "-", alpha = 0.7)
# axs[2].annotate('E-cell start \n18% increase',(4060,120), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
axs[2].annotate('13% increase',(3970,170), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
loc = plticker.MultipleLocator(base=2000)
axs[2].xaxis.set_major_locator(loc)
axs[2].axvspan(-250,3960, alpha=0.1, color='#4574A2FF')


# Hide x labels and tick labels for top plots and y ticks for right plots.
for ax in axs.flat:
    ax.label_outer()

plt.savefig("SigmaComp.png", format="png", bbox_inches="tight")


-----------------------------------
# old plots

# FOM and window plot

In [ ]:
fig, axs = plt.subplots(1, 2, figsize = (15,7))
fig.tight_layout(pad=3)

# axs.errorbar(id325og['binTime(s)'], id325og['Neutrons(cps)'], id325og['delta-neutrons']*1.5, ls ='', marker = 's', ms = 3, capsize = 3, label = 'ID-325', color ='#4574A2FF' )
axs[1].plot(psdData['Energy'], psdData['PSD'], ls ='', marker = 'o', ms = 2, alpha = 0.1, color = 'black' )
axs[1].set_ylabel('PSD (a. u.)', fontsize=16)
axs[1].annotate('Neutron channel',(250,0.315), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, fontname = 'Arial', color = 'white' )
axs[1].annotate('Gamma-ray channel',(250,0.160), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, fontname = 'Arial',color = 'white' )
axs[1].set_xlabel('Energy (keVee)', fontsize=16)
axs[1].tick_params(axis="x", labelsize = 14, )
axs[1].tick_params(axis="y", labelsize = 14)
axs[1].plot(xaxis, yaxis,  ls ='-', color = 'C1')
axs[1].plot(xaxis, yaxis+0.2,  ls ='-', color = 'C1')
axs[1].axvline(195, ls = '--', color = 'C0')
threshold = 195
axs[1].axhline(0.155, color = 'white')
axs[1].fill_between(xaxis, yaxis,yaxis+0.2, where=xaxis>threshold, color = 'C1', alpha = 0.3)

# axs[0].plot(data.iloc[13].index, data.iloc[13].values)

axs[0].set_xlabel('Energy (keVee)', fontsize = 16)
axs[0].set_ylabel('FOM', fontsize = 16)
axs[0].plot(newFOM['i']*15, newFOM['fom'], ls ='-', marker = 's', ms = 0, color = 'black')
axs[0].axhline(1.27, ls = '--', color = 'C0')
axs[0].annotate('FOM = 1.27',(50,1.29), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, fontname = 'Arial',color = 'black' )
# axs[0].set_title('Slice #13 - 195 keVee', fontsize = 14)

axs[0].set_xlim(-10, 600)
axs[0].set_ylim(0.25, 2)
axs[0].tick_params(axis="x", labelsize = 14)
axs[0].tick_params(axis="y", labelsize = 14)

plt.show()


In [ ]:
fig, axs = plt.subplots(1, 2, figsize = (15,7))
fig.tight_layout(pad=2)

# axs.errorbar(id325og['binTime(s)'], id325og['Neutrons(cps)'], id325og['delta-neutrons']*1.5, ls ='', marker = 's', ms = 3, capsize = 3, label = 'ID-325', color ='#4574A2FF' )
axs[0].plot(psdData['Energy'], psdData['PSD'], ls ='', marker = 'o', ms = 3, alpha = 0.2 )
axs[0].plot(FOM['e_lower'], FOM['lowBound'], marker ='o', ls ='')
axs[0].plot(FOM['e_lower'], FOM['highBound'], ls ='-')
axs[0].plot(xaxis, yaxis)
# axs[1].plot(FOM['e_lower'], FOM['fom'], ls ='', marker = 's', ms = 3, alpha = 0.3 )
# axs.errorbar(id378['binTime(s)'], id378['scaledCounts'], id378['delta-neutrons'], ls ='', marker = 's', ms = 3, capsize = 3, label = 'ID-375 - Target A4', color ='#424242FF' )
# axs.set_title('Beam Loading', fontsize = 16)
# axs.legend(fontsize = 12, loc = 'lower right')
# axs.set_xlim(-360, 2000)
# axs[0].set_ylim(-0.04,0.54)
axs[0].set_ylabel('PSD (a. u.)', fontsize=14)
axs[0].annotate('Neutron channel',(250,0.315), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, fontname = 'Arial', color = 'white' )
axs[0].annotate('Gamma-ray channel',(250,0.15), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14, fontname = 'Arial',color = 'white' )
axs[0].set_xlabel('Energy (keVee)', fontsize=14)
# axs[0].axhline(0.35, color = 'black', ls = "--", alpha = 0.7)
axs[0].axhline(0.16, color = 'black', ls = "--", alpha = 0.7)
axs[0].axvline(195,color = 'black', ls ='--')
axs[0].axvline(210,color = 'black', ls ='--')
# axs[0].axvline(100, color = 'black')
axs[0].tick_params(axis="x", labelsize = 12)

axs[1].plot(data.iloc[13].index, data.iloc[13].values)
axs[1].set_xlabel('PDS (a.u.)', fontsize = 14)
axs[1].set_ylabel('Counts', fontsize = 14)
axs[1].axvline(30, ls = '--', color = 'black')
axs[1].set_title('Slice #13 - 195 keVee', fontsize = 14)

plt.show()


## Energy calibration plot

In [ ]:
def readCalData(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=1)
    tmp.columns = ['Isotope','EnergyPeak', 'LightOutput', 'ADCChannel']
    return tmp

In [ ]:
calData = readCalData('calibrationcurve.csv')

In [ ]:
csData = calData.iloc[0]
coData = calData.iloc[1:3]
euData = calData.iloc[3:8]

In [ ]:
x = np.linspace(0, 2600, 100)

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (7,7))
fig.tight_layout(pad=2)

# axs.errorbar(id325og['binTime(s)'], id325og['Neutrons(cps)'], id325og['delta-neutrons']*1.5, ls ='', marker = 's', ms = 3, capsize = 3, label = 'ID-325', color ='#4574A2FF' )
axs.plot(coData['ADCChannel'], coData['LightOutput'], ls ='', marker = 'o', ms = 8, color = 'blue', label = '$^{60}$Co')
axs.plot(csData['ADCChannel'], csData['LightOutput'], ls ='', marker = 's', ms = 6, color = 'black', label = '$^{137}$Cs')
axs.plot(euData['ADCChannel'], euData['LightOutput'], ls ='', marker = '^', ms = 8, color = 'red', label = '$^{152}$Eu')
axs.plot(x, (x-41.67)/2.179, linestyle='--', alpha = 0.5, label = 'Fit') 
# axs.errorbar(id378['binTime(s)'], id378['scaledCounts'], id378['delta-neutrons'], ls ='', marker = 's', ms = 3, capsize = 3, label = 'ID-375 - Target A4', color ='#424242FF' )
# axs.set_title('Beam Loading', fontsize = 16)
axs.legend(fontsize = 14, loc = 'lower right')
# axs.set_xlim(-33,1700)
# axs.set_ylim(-0.04,0.7)
axs.set_xlabel('Analog-to-Digital Converter channel (arb. units)', fontsize=16)
axs.set_ylabel('Light output (keVee)', fontsize=16)

axs.tick_params(axis="x", labelsize = 14)
axs.tick_params(axis="y", labelsize = 14)
# axs.axvline(195, color = 'red')
# axs.axvline(205, color = 'red')
# axs.plot(xaxis, yaxis,  ls ='--', color = 'C1')
# axs.plot(xaxis, yaxis+0.2,  ls ='--', color = 'C1')
# axs.fill_between(xaxis, yaxis,yaxis+0.2, where=xaxis>threshold, color = 'C1', alpha = 0.3)

plt.savefig("GammaCalibration.png", format="png", bbox_inches="tight")

plt.show()

